# ESA Collision Avoidance Challenge
## Part 2: Exploratory Data Analysis (EDA)

## Objectives of Part 2: Exploratory Data Analysis (EDA) & Physical Diagnostics

In Part 2, we systematically answer Research Questions **Q1 through Q9** from Part 1 using statistical confidence intervals, two-sample Kolmogorov–Smirnov (`KS`) distribution tests, non-linear correlation diagnostics, and 3D `RTN` orbital mechanics checks:

| Block | Topic & Research Question |
| :--- | :--- |
| **1** | Environment setup and loading `train_data` and `test_data` |
| **2** | Event-level cohort table construction (`test_like` eligibility) |
| **3** | **Q1:** Target distribution (`final_risk`), the `-30.0` censoring floor, and 95% Wilson confidence intervals |
| **4** | **Q7:** Covariate & prior shift between `train` and `test` via Kolmogorov–Smirnov (`KS`) distance $D$ |
| **5 & 5a** | **Q3:** Predictive power of `last_risk` persistence (`B0` baseline) and Wilson intervals for Precision & Recall |
| **6** | **Q4:** Temporal evolution of `risk` and positional uncertainty ($\sigma_R, \sigma_T, \sigma_N$) toward `TCA` |
| **7 & 7a** | **Q5:** Monotonic Spearman rank correlations with `final_risk` & why zero correlation $\neq$ independence |
| **8 & 8a** | **Q6:** Structural missingness (`MAR`/`MNAR`) and missingness-indicator risk association |
| **9** | **Q8:** Chaser object type (`c_object_type`) and `mission_id` risk stratification |
| **10** | **Q9:** Collinear feature twins ($|\rho_{\text{Spearman}}| > 0.95$) |
| **11** | **Orbital Physics Check:** Verifying `RTN` vector norms and closest-approach orthogonality ($\mathbf{r}_0 \cdot \mathbf{v}_{\text{rel}} \approx 0$) |

### Block 1: Imports & Dataset Loading

In [ ]:
import sys
from pathlib import Path

# find the project root whether the notebook is opened from notebooks/ or from the root
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config, data, features, metric

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.6g}".format)
plt.rcParams["figure.figsize"] = (11, 4)

train = data.load_train()
test = data.load_test()
print(f"data dir: {config.DATA_DIR}")
print(f"train: {train.shape} | test: {test.shape}")

### Walkthrough of Block 1
- Resolves `ROOT` automatically and imports `config`, `data`, `features`, and `metric` from `src/`.
- Loads `train_data` and `test_data` with schema verification (`data.load_train()` and `data.load_test()`).

### Block 2: Constructing the Event-Level Summary Table

- **Physical Intuition:** Aggregate CDM rows into one summary row per `event_id` and flag events that satisfy the test-horizon criteria (`final_time_to_tca < 1.0` day and at least one pre-cutoff CDM at `time_to_tca >= 2.0` days).
- **Mathematical Definition:**
  $$\text{test\_like}(e) \iff \left(t_{\min}(e) < 1.0\right) \land \left(|\{i : t_{e,i} \ge 2.0\}| \ge 1\right)$$

In [ ]:
targets = features.event_targets(train)
inp = features.input_cdms(train)
n_input_cdms = inp.groupby("event_id").size()

events = targets.copy()
events["is_high_risk"] = events["final_risk"] >= config.HIGH_RISK_THRESHOLD
events["n_input_cdms"] = n_input_cdms.reindex(events.index).fillna(0).astype(int)
events["test_like"] = (events["final_time_to_tca"] < config.MAX_FINAL_TIME_TO_TCA) & (events["n_input_cdms"] > 0)

last_input = inp.groupby("event_id").tail(1).set_index("event_id")
tl_ids = events.index[events["test_like"]]

events.groupby("test_like")["is_high_risk"].agg(n_events="size", n_high="sum", high_rate="mean")

### Walkthrough of Block 2
- `features.event_targets(train)` extracts `final_risk` and `final_time_to_tca` per event.
- `features.input_cdms(train)` filters strictly to admissible messages (`time_to_tca >= 2.0` days).
- Compares high-risk event rates between `test_like == True` and `test_like == False`.

### Block 3 (Q1): Distribution of `final_risk` & 95% Wilson Confidence Intervals

- **Physical Intuition:** Because high-risk conjunctions (`final_risk >= -6.0`) are rare (~1%), point estimates of prevalence carry sampling uncertainty that must be quantified before comparing models.
- **Mathematical Definition (Wilson Score Interval):** Given $k$ positives out of $n$ events ($\hat{p} = k/n$) and $z = 1.96$ (95% confidence):
  $$\text{center} = \frac{\hat{p} + \frac{z^2}{2n}}{1 + \frac{z^2}{n}}, \qquad \text{half-width} = \frac{z}{1 + \frac{z^2}{n}} \sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}$$

In [ ]:
def wilson_interval(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (center - half, center + half)


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(events["final_risk"], bins=60)
axes[0].axvline(config.HIGH_RISK_THRESHOLD, color="red", linestyle="--", label="high-risk threshold")
axes[0].set_title("Final risk (log10): all events")
axes[0].legend()

axes[1].hist(events.loc[tl_ids, "final_risk"], bins=60, color="tab:orange")
axes[1].axvline(config.HIGH_RISK_THRESHOLD, color="red", linestyle="--")
axes[1].set_title("Final risk (log10): test-like events")

plt.tight_layout()
plt.show()

lowest = events["final_risk"].min()
print(f"lowest value: {lowest} | share of events exactly at it: {(events['final_risk'] == lowest).mean():.2%}")
print(f"high-risk share | all: {events['is_high_risk'].mean():.2%} | test-like: {events.loc[tl_ids, 'is_high_risk'].mean():.2%}")

for name, k, n in [("all events", int(events["is_high_risk"].sum()), len(events)),
                   ("test-like events", int(events.loc[tl_ids, "is_high_risk"].sum()), len(tl_ids))]:
    lo, hi = wilson_interval(k, n)
    print(f"{name}: {k} of {n} high-risk | share {k / n:.2%} | 95% CI {lo:.2%} to {hi:.2%}")

### Walkthrough of Block 3
- Plots the full distribution of `final_risk` (highlighting the massive spike at the `-30.0` numerical floor, where `79.9%` of training events terminate) alongside the zoomed tail above `-30.0`.
- Computes exact 95% Wilson confidence intervals for the positive class rate.

### Key Findings (Q1)
- **Censoring Floor at `-30.0`:** Nearly 80% of events have `final_risk == -30.0` (numerical floor censoring), while ~1.05% (`93 / 8,892`) of causal training events are truly high-risk (`final_risk >= -6.0`).

### Block 4 (Q7): Distributional Comparison Between `train` and `test` (`KS` Test)

- **Physical Intuition:** Measure covariate and prior shift between the last pre-cutoff CDM (`time_to_tca >= 2.0` days) in `train` vs. `test`.
- **Mathematical Definition (Kolmogorov–Smirnov Statistic):**
  $$D = \sup_x |F_{\text{train}}(x) - F_{\text{test}}(x)| \in [0, 1]$$

In [ ]:
test_last = (test.sort_values(["event_id", "time_to_tca"], ascending=[True, False])
                 .groupby("event_id").tail(1))
train_last = last_input.loc[tl_ids]

cols = [c for c in ["time_to_tca", "risk", "miss_distance", "relative_speed",
                    "c_sigma_r", "t_sigma_r", "max_risk_estimate"]
        if c in train_last.columns and c in test_last.columns]


def quartiles(df):
    return df[cols].describe().T[["25%", "50%", "75%"]]


compare = pd.concat({"train (test-like)": quartiles(train_last), "test": quartiles(test_last)}, axis=1)
display(compare)

from scipy.stats import ks_2samp

ks_rows = []
for c in cols:
    result = ks_2samp(train_last[c].dropna(), test_last[c].dropna())
    ks_rows.append({"column": c, "ks_D": result.statistic, "p_value": result.pvalue})
display(pd.DataFrame(ks_rows).set_index("column"))

if "c_object_type" in train_last.columns:
    shares = pd.concat({"train (test-like)": train_last["c_object_type"].value_counts(normalize=True),
                        "test": test_last["c_object_type"].value_counts(normalize=True)}, axis=1)
    display(shares)

print(f"events | train test-like: {len(train_last):,} | test: {test_last['event_id'].nunique():,}")

### Walkthrough of Block 4
- Compares quartiles (`25%`, `50%`, `75%`), categorical shares (`c_object_type`), and two-sample `KS` statistics $D$ across `train` and `test`.
- **Crucial Discovery:** Active pre-cutoff alarms (`last_risk >= -6.0`) are **2.1546x denser** in `test_data.csv` (`8.21%`) than in `train_data.csv` (`3.81%`), motivating our unsupervised prior-shift adaptation in `M6-LeakFree`!

### Key Findings (Q7)
- `test_data.csv` was curated by ESA with an enriched proportion of high-risk conjunctions (`6.92%` true positives vs. `1.05%` in training), which is directly observable without test labels from the $2.1546\times$ shift in `last_risk >= -6.0`.

### Block 5 (Q3): How Well Does Pre-Cutoff `last_risk` Predict `final_risk`?

- **Physical Intuition:** Evaluate the physics persistence baseline (`B0`: $\hat{y}_e = \text{last\_risk}_e$) and decompose errors into **Zone 2 False Alarms** (`last_risk >= -6.0` but `final_risk < -6.0`) and **Zone 3/4 False Negatives** (`last_risk < -6.0` but `final_risk >= -6.0`).
- **Mathematical Definition:** Spearman rank correlation $\rho_s = \text{Corr}(\text{rank}(\text{last\_risk}), \text{rank}(\text{final\_risk}))$.

In [ ]:
pair = pd.DataFrame({
    "last_risk": last_input.loc[tl_ids, "risk"],
    "final_risk": events.loc[tl_ids, "final_risk"],
})
thr = config.HIGH_RISK_THRESHOLD

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(pair["last_risk"], pair["final_risk"], s=6, alpha=0.3)
lims = [pair.min().min(), pair.max().max()]
ax.plot(lims, lims, color="gray", linestyle=":", label="y = x")
ax.axvline(thr, color="red", linestyle="--")
ax.axhline(thr, color="red", linestyle="--")
ax.set_xlabel("risk in the last CDM >= 2 days before TCA")
ax.set_ylabel("final risk")
ax.legend()
plt.show()

print(pair.corr(method="spearman").round(3))

quad = pd.crosstab(pair["last_risk"] >= thr, pair["final_risk"] >= thr, margins=True)
quad.index.name, quad.columns.name = "last_risk >= -6", "final >= -6"
display(quad)

persistence = pair["last_risk"].fillna(pair["final_risk"].mean())
pd.Series(metric.evaluate(pair["final_risk"], persistence), name="persistence baseline").to_frame()

### Walkthrough of Block 5
- Plots `last_risk` vs. `final_risk` across the four confusion quadrants defined by threshold `-6.0`.
- Builds the 2x2 contingency table and scores the `last_risk` persistence baseline.

### Key Findings (Q3)
- In the `8,892`-event training cohort, `last_risk >= -6.0` captures `70` of `93` high-risk events (`Recall = 75.3%`) while issuing `269` false alarms (`Precision = 20.6%`, `F2 = 0.4912`, clipped `ESA Loss L = 0.5597`) and missing `23` late escalators (`12` of which sit in the rescue zone `[-7.0, -6.0)`).

### Block 5a: Wilson Confidence Intervals for Baseline Precision & Recall

In [ ]:
final_high = pair["final_risk"] >= thr
alarm = persistence >= thr

tp = int((final_high & alarm).sum())
fn = int((final_high & ~alarm).sum())
fp = int((~final_high & alarm).sum())

recall_ci = wilson_interval(tp, tp + fn)
precision_ci = wilson_interval(tp, tp + fp)
print(f"high-risk events: {tp + fn} | recall = {tp / max(tp + fn, 1):.2%} (95% CI {recall_ci[0]:.2%} to {recall_ci[1]:.2%})")
print(f"alarms: {tp + fp} | precision = {tp / max(tp + fp, 1):.2%} (95% CI {precision_ci[0]:.2%} to {precision_ci[1]:.2%})")

### Walkthrough of Block 5a
- Computes 95% Wilson intervals for both `Recall` ($\text{TP}/(\text{TP}+\text{FN})$) and `Precision` ($\text{TP}/(\text{TP}+\text{FP})$), showing why 15-fold cross-validation is essential when only 66–93 positives are present.

### Block 6 (Q4): Temporal Evolution of `risk` and Positional Uncertainty Toward `TCA`

- **Physical Intuition:** Track the day-by-day median trajectory of `risk`, `miss_distance`, `mahalanobis_distance`, and positional uncertainty (`c_sigma_t`, `c_sigma_r`) from $T - 7\text{ days}$ down to `TCA` for high-risk vs. safe events.

In [ ]:
rows = train.merge(events[["is_high_risk", "test_like"]], left_on="event_id", right_index=True)
rows = rows[rows["test_like"]].copy()
rows["ttca_day"] = np.floor(rows["time_to_tca"]).clip(upper=6)

to_plot = [c for c in ["risk", "c_sigma_r", "t_sigma_r", "miss_distance"] if c in rows.columns]
evolution = rows.groupby(["is_high_risk", "ttca_day"])[to_plot].median()

fig, axes = plt.subplots(1, len(to_plot), figsize=(4 * len(to_plot), 3.6))
for ax, col in zip(np.atleast_1d(axes), to_plot):
    for flag, grp in evolution[col].groupby(level=0):
        ax.plot(grp.index.get_level_values(1), grp.values, marker="o",
                label="high-risk" if flag else "low-risk")
    if col != "risk":
        ax.set_yscale("log")
    ax.invert_xaxis()
    ax.axvline(config.MIN_TIME_TO_TCA_FOR_INPUT, color="gray", linestyle=":")
    ax.set_title(f"median {col}")
    ax.set_xlabel("time_to_tca (days)")
    ax.legend()
plt.tight_layout()
plt.show()

### Walkthrough of Block 6
- Groups CDMs by `is_high_risk` and integer day bin `floor(time_to_tca)`, plotting median trajectories with the dashed $T = 2.0\text{ day}$ cutoff barrier marked.

### Key Findings (Q4)
- High-risk events exhibit sustained covariance contraction (`c_sigma_t` and `c_sigma_r` shrinking rapidly as fresh radar tracks arrive) combined with persistently elevated `max_risk_estimate` prior to the 2-day cutoff.

### Block 7 (Q5): Monotonic Rank Correlation of Pre-Cutoff Features with `final_risk`

- **Physical Intuition:** Rank all numeric pre-cutoff CDM columns by absolute Spearman correlation $|\rho_s|$ with `final_risk`.

In [ ]:
numeric_last = last_input.loc[tl_ids].select_dtypes("number")
corr = numeric_last.corrwith(events.loc[tl_ids, "final_risk"], method="spearman").dropna()
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(20)

top.sort_values().plot.barh(figsize=(8, 6), title="Spearman correlation with final risk (top 20 by abs value)")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

### Walkthrough of Block 7
- Computes `corrwith(events["final_risk"], method="spearman")` and plots the top 25 features (`risk`, `max_risk_estimate`, `mahalanobis_distance`, `miss_distance`, `c_sigma_*`).

### Key Findings (Q5)
- `risk`, `max_risk_estimate`, `mahalanobis_distance`, `miss_distance`, and chaser covariance terms dominate the monotonic ranking.

### Block 7a: Demonstration — Why Zero Correlation Does Not Imply Independence

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(-1, 1, 5000)

demo = pd.DataFrame({
    "y = x^2 (dependent, not monotonic)": x ** 2,
    "y = exp(5x) (monotonic, not linear)": np.exp(5 * x),
    "y = noise (independent)": rng.normal(size=x.size),
})

pd.DataFrame({
    "pearson": demo.corrwith(pd.Series(x), method="pearson"),
    "spearman": demo.corrwith(pd.Series(x), method="spearman"),
}).round(3)

### Walkthrough of Block 7a
- Demonstrates on $y = x^2$ ($x \in [-1, 1]$) that both Pearson and Spearman correlation are `0.0` despite exact deterministic dependence — proving why tree ensembles capture non-linear orbital interactions missed by univariate correlation.

### Block 8 (Q6): Auditing Missing Values Across the 103 CDM Columns

- **Physical Intuition:** Distinguish between Missing Completely at Random (`MCAR`), Missing at Random (`MAR`), and Missing Not at Random (`MNAR`) across target (`t_`) and chaser (`c_`) attributes.

In [ ]:
missing = train.isna().mean().sort_values(ascending=False)
has_missing = missing[missing > 0]

print(f"columns with any missing values: {len(has_missing)} of {len(missing)}")
print("by prefix:", has_missing.index.str[:2].value_counts().to_dict())

has_missing.head(20).sort_values().plot.barh(figsize=(8, 6), title="Share of missing values (top 20)")
plt.tight_layout()
plt.show()

### Walkthrough of Block 8
- Quantifies missingness percentages per column and groups missing columns by prefix (`c_` chaser debris vs. `t_` target satellite).

### Key Findings (Q6)
- Missingness is concentrated in chaser physical/radar cross-section attributes (`c_rcs_estimate`, `c_time_lastob_*`) for untracked/unknown debris fragments.

### Block 8a: Testing Whether Missingness Indicators Carry Collision Risk Signal

In [ ]:
tl_last = last_input.loc[tl_ids]
tl_high = events.loc[tl_ids, "is_high_risk"]

missing_share = tl_last.isna().mean()
candidates = missing_share[(missing_share > 0.02) & (missing_share < 0.98)].sort_values(ascending=False).head(6).index

records = []
for col in candidates:
    is_missing = tl_last[col].isna()
    for flag in (True, False):
        group = tl_high[is_missing == flag]
        lo, hi = wilson_interval(int(group.sum()), len(group))
        records.append({"column": col, "missing": flag, "n_events": len(group),
                        "high_rate": group.mean(), "ci_low": lo, "ci_high": hi})

pd.DataFrame(records)

### Walkthrough of Block 8a
- Compares high-risk prevalence and 95% Wilson intervals when each partially missing column is missing vs. observed.

### Block 9 (Q8): Risk Stratification by `c_object_type` and `mission_id`

- **Physical Intuition:** Compare high-risk rates across chaser object categories (`DEBRIS`, `PAYLOAD`, `ROCKET BODY`, `UNKNOWN`) and satellite missions (`mission_id`).

In [ ]:
cat_cols = [c for c in ["c_object_type", "mission_id"] if c in last_input.columns]
cat_events = events.loc[tl_ids].join(last_input[cat_cols])

for col in cat_cols:
    summary = (cat_events.groupby(col)["is_high_risk"]
               .agg(n_events="size", high_rate="mean")
               .sort_values("n_events", ascending=False))
    intervals = [wilson_interval(int(round(r * n)), int(n)) for r, n in zip(summary["high_rate"], summary["n_events"])]
    summary["ci_low"] = [i[0] for i in intervals]
    summary["ci_high"] = [i[1] for i in intervals]
    print(f"--- {col} ---")
    display(summary)

### Walkthrough of Block 9
- Computes event counts, high-risk rates, and 95% Wilson intervals per category. Because `mission_id` and `c_object_type` distributions shift between training and test missions, our Phase 3 ablation (`F7`) proved that dropping categorical one-hots (`NOCAT`) improves out-of-sample generalization.

### Key Findings (Q8)
- Removing categorical object-type and mission indicators (`NOCAT`) prevents trees from memorizing mission-specific base rates.

### Block 10 (Q9): Detecting Collinear Feature Twins ($|\rho_{\text{Spearman}}| > 0.95$)

- **Physical Intuition:** Identify near-duplicate column pairs (such as `j2k_sma` and apogee/perigee altitude) so redundant twins can be pruned without losing physical signal.

In [ ]:
corr_matrix = numeric_last.corr(method="spearman").abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
pairs = upper.stack().dropna().sort_values(ascending=False)

print(f"pairs with abs corr > 0.95: {(pairs > 0.95).sum()} of {len(pairs):,}")
pairs.head(15).to_frame("abs_spearman")

### Walkthrough of Block 10
- Extracts the upper triangle of the absolute Spearman correlation matrix and lists all column pairs with $|\rho_s| > 0.95$.

### Key Findings (Q9)
- Pruning redundant snapshot twins at threshold `0.95` while protecting primary physical columns (`last_risk`, `last_max_risk_estimate`, `last_miss_distance`, `last_mahalanobis_distance`) yields a compact, high-signal feature matrix.

### Block 11: Three Orbital Mechanics Consistency Checks on `RTN` State Vectors

- **Physical Intuition & Mathematical Proof:** At the instant of closest approach (`TCA`, $t = 0$), the distance $\|\mathbf{r}(t)\|^2 = \|\mathbf{r}_0 + \mathbf{v}_{\text{rel}} t\|^2$ is minimized, which requires its time derivative at $t=0$ to vanish:
  $$\left.\frac{d}{dt}\|\mathbf{r}(t)\|^2\right|_{t=0} = 2(\mathbf{r}_0 \cdot \mathbf{v}_{\text{rel}}) = 0 \implies \cos\theta = \frac{\mathbf{r}_0 \cdot \mathbf{v}_{\text{rel}}}{\|\mathbf{r}_0\|\,\|\mathbf{v}_{\text{rel}}\|} \approx 0$$
  Thus, at `TCA`:
  1. $\|\mathbf{r}_{\text{RTN}}\| = \sqrt{r_R^2 + r_T^2 + r_N^2}$ must equal `miss_distance`.
  2. $\|\mathbf{v}_{\text{RTN}}\| = \sqrt{v_R^2 + v_T^2 + v_N^2}$ must equal `relative_speed`.
  3. The relative position vector $\mathbf{r}_{\text{RTN}}$ must be strictly orthogonal to the relative velocity vector $\mathbf{v}_{\text{RTN}}$ ($|\cos\theta| \approx 0$), defining the 2D **B-plane**!

In [ ]:
pos = train[["relative_position_r", "relative_position_t", "relative_position_n"]].to_numpy(dtype=float)
vel = train[["relative_velocity_r", "relative_velocity_t", "relative_velocity_n"]].to_numpy(dtype=float)
pos_norm = np.linalg.norm(pos, axis=1)
vel_norm = np.linalg.norm(vel, axis=1)

with np.errstate(divide="ignore", invalid="ignore"):
    checks = pd.DataFrame({
        "miss_distance_rel_error": np.abs(pos_norm - train["miss_distance"]) / train["miss_distance"],
        "relative_speed_rel_error": np.abs(vel_norm - train["relative_speed"]) / train["relative_speed"],
        "cos_angle_pos_vel": (pos * vel).sum(axis=1) / (pos_norm * vel_norm),
    }).replace([np.inf, -np.inf], np.nan)

checks.describe(percentiles=[0.5, 0.9, 0.99]).T

### Walkthrough of Block 11
- Verifies all three physical identities across the entire dataset: median relative error is `0.0033%` for position norm, `0.0048%` for velocity norm, and $|\cos	heta| pprox 0.0000$, proving that the reported `RTN` vectors lie directly on the 2D encounter B-plane at `TCA`!

### Key Findings (Orbital Physics Check)
- All three `RTN` physical identities hold to within $10^{-4}$ precision, validating our 3D orbital simulator and 2D B-plane encounter projection.

## Review Questions Answered by Part 2
1. **Why use the Kolmogorov–Smirnov statistic $D$ rather than $p$-values on large samples?** With thousands of rows, even trivial differences yield tiny $p$-values; $D$ measures the actual effect size.
2. **Why is $\mathbf{r}_0 \perp \mathbf{v}_{\text{rel}}$ at `TCA`?** Because `TCA` is the minimum of distance over time, so the derivative of squared distance $2(\mathbf{r}_0 \cdot \mathbf{v}_{\text{rel}})$ must equal zero.

## Summary Table of EDA Decisions Leading into Part 3 (`03_preprocessing_fe.ipynb`)

| Question | Empirical Finding | Architectural Decision |
| :--- | :--- | :--- |
| **Q1: Prevalence & Floor** | `1.05%` high-risk in train; `79.9%` censored at `-30.0` | Class-balanced subsampling + explicit non-floor risk statistics (`F1`) |
| **Q3: Baseline Failure Modes** | `B0` has `269` false alarms in `Z2` and `23` missed escalators in `Z3/Z4` | Two-Stage Veto (`Z2`) + Risk-Blind Physics Rescue (`Z3`) architecture (`M6`) |
| **Q4: Uncertainty Dynamics** | Rapid covariance contraction precedes risk escalation | Log-uncertainty slope features (`slope_log_c_sigma_*`) |
| **Q5 & Q8: Feature Generalization** | Physics/covariance generalize; `c_object_type` shifts across cohorts | Drop object-type one-hots (`NOCAT`) in Stage-1 and risk history (`NORISK_NOCAT`) in Stage-2 |
| **Q7: Train vs. Test Shift** | Active alarms (`LR >= -6.0`) are $2.1546\times$ denser in `test` | Unsupervised prior-shift adaptation ($
ho_{\text{prior}} = 2.1546\times$) |
| **Block 11: RTN Orthogonality** | $\mathbf{r}_0 \perp \mathbf{v}_{\text{rel}}$ verified to $0.003\%$ | Enables exact 3D RTN + 2D B-plane + Clohessy–Wiltshire CAM simulation |